# Лабораторная работа №1
## Предобработка данных. Знакомство с pandas
**Датасет:** Заболевания щитовидной железы (Sick Dataset)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Загрузка данных и вывод на экран

In [ ]:
df = pd.read_csv('sick_dataset.csv')
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов')
display(df.head())

## 2. Количество пропущенных значений для каждого столбца

In [ ]:
print('=== Пропущенные значения ДО заполнения ===')
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

## 3. Заполнение пропущенных значений
Столбец `TBG` пуст полностью (3772 пропусков), поэтому мы его просто удалим. Остальные заполним медианой (для чисел) и модой (для строк).

In [ ]:
# Удаляем полностью пустой столбец TBG
df.drop(columns=['TBG'], inplace=True)

# Проходимся по всем столбцам и заполняем пропуски
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

print('\n=== Пропущенные значения ПОСЛЕ заполнения ===')
print(f'Осталось пустых ячеек во всем датасете: {df.isnull().sum().sum()}')

## 4. Нормализация данных
Применяем MinMaxScaler к числовым столбцам.

In [ ]:
# Выделяем числовые столбцы
numeric_cols = df.select_dtypes(include='number').columns

scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print('\n=== Данные после нормализации ===')
display(df[numeric_cols].head())

## 5. Преобразование категориальных данных (One-Hot Encoding)

In [ ]:
object_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=object_cols, drop_first=True)

print(f'\nИтоговый размер датасета после OHE: {df.shape}')
display(df.head())

## Разделение на выборки и сохранение

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
print(f'Обучающая выборка: {train_df.shape}')
print(f'Тестовая выборка: {test_df.shape}')

df.to_csv('processed_sick_dataset.csv', index=False)
print('\nОбработанный датасет сохранён!')